# M21C LS Unified Paper Figures

This notebook is the paper-figure control room. It starts with shared paths, period definitions, and input-file availability checks only. Plotting sections should be added after these checks are clean enough for the figure being rebuilt.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / ".git").exists() and (p / "common/python/io/read_GEOSldas.py").exists():
            return p
    raise FileNotFoundError("Could not locate geosldas-analysis repo root")


HERE = Path.cwd().resolve()
REPO_ROOT = find_repo_root(HERE)
PROJECT_ROOT = REPO_ROOT / "projects/M21C_ls"
PAPER_FIG_DIR = PROJECT_ROOT / "output/paper_figures"
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)

GEOSLDAS_DIAG_ROOT = Path("/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2")
DISCOVER_REPO_ROOT = Path("/discover/nobackup/projects/land_da/geosldas-analysis")

print("REPO_ROOT:", REPO_ROOT)
print("PAPER_FIG_DIR:", PAPER_FIG_DIR)
print("GEOSLDAS_DIAG_ROOT exists:", GEOSLDAS_DIAG_ROOT.exists())

## Period Registry

Use fine-grain observing-system periods for timeline/time-series/OFA figures, and three broader validation periods for ISMN and ERA5-Land summaries.

In [ ]:
PAPER_START = pd.Timestamp("2000-06-01")
PAPER_END = pd.Timestamp("2024-05-31")

fine_periods = pd.DataFrame([
    {"period_id": "P1", "start": "2000-06-01", "end": "2002-06-30", "label": "MODIS Terra SCF", "validation_id": "V1"},
    {"period_id": "P2", "start": "2002-07-01", "end": "2007-05-31", "label": "MODIS Terra+Aqua SCF", "validation_id": "V1"},
    {"period_id": "P3", "start": "2007-06-01", "end": "2010-04-30", "label": "SCF + ASCAT-A", "validation_id": "V2"},
    {"period_id": "P4", "start": "2010-05-01", "end": "2013-03-31", "label": "SCF + ASCAT-A + SMOS", "validation_id": "V2"},
    {"period_id": "P5", "start": "2013-04-01", "end": "2015-03-31", "label": "SCF + ASCAT-A/B + SMOS", "validation_id": "V2"},
    {"period_id": "P6", "start": "2015-04-01", "end": "2018-07-31", "label": "SCF + ASCAT-A/B + SMOS + SMAP", "validation_id": "V3"},
    {"period_id": "P7", "start": "2018-08-01", "end": "2019-10-31", "label": "SCF + ASCAT-A/B + SMOS + SMAP + CYGNSS", "validation_id": "V3"},
    {"period_id": "P8", "start": "2019-11-01", "end": "2021-11-30", "label": "SCF + ASCAT-A/B/C + SMOS + SMAP + CYGNSS", "validation_id": "V3"},
    {"period_id": "P9", "start": "2021-12-01", "end": "2024-05-31", "label": "SCF + ASCAT-B/C + SMOS + SMAP + CYGNSS", "validation_id": "V3"},
])

validation_periods = pd.DataFrame([
    {"period_id": "V1", "start": "2000-06-01", "end": "2007-05-31", "label": "SCF-only era", "fine_periods": "P1 + P2"},
    {"period_id": "V2", "start": "2007-06-01", "end": "2015-03-31", "label": "pre-SMAP microwave era", "fine_periods": "P3 + P4 + P5"},
    {"period_id": "V3", "start": "2015-04-01", "end": "2024-05-31", "label": "SMAP-era multi-sensor era", "fine_periods": "P6 + P7 + P8 + P9"},
])

for df in (fine_periods, validation_periods):
    df["start"] = pd.to_datetime(df["start"])
    df["end"] = pd.to_datetime(df["end"])
    df["n_days_inclusive"] = (df["end"] - df["start"]).dt.days + 1

assert fine_periods["start"].iloc[0] == PAPER_START
assert fine_periods["end"].iloc[-1] == PAPER_END
assert validation_periods["start"].iloc[0] == PAPER_START
assert validation_periods["end"].iloc[-1] == PAPER_END

display(fine_periods)
display(validation_periods)

## Figure And Data Registry

`required` files are needed to rebuild the figure from data. `reference` files are existing figure products useful for visual comparison.

In [ ]:
def p_rel(path: str) -> Path:
    return REPO_ROOT / path


def p_diag(path: str) -> Path:
    return GEOSLDAS_DIAG_ROOT / path


def p_discover(path: str) -> Path:
    return DISCOVER_REPO_ROOT / path


figure_registry = [
    {
        "figure": "Fig. 1",
        "description": "Observing-system timeline",
        "period_set": "fine_periods",
        "required": [],
        "reference": [],
        "upstream": "Period registry in this notebook",
    },
    {
        "figure": "Fig. 2",
        "description": "Mean assimilated observations per day",
        "period_set": "full record",
        "required": [
            p_diag("temporal_stats_DA_20000601_20240531.nc4"),
            p_diag("spatial_stats_DA_200006_202405.pkl"),
            p_diag("LS_OLv8_M36.ldas_tilecoord.bin"),
        ],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 3",
        "description": "Monthly assimilated observation counts",
        "period_set": "fine_periods",
        "required": [p_diag("spatial_stats_DA_200006_202405.pkl")],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 4",
        "description": "Full-period OmF maps by sensor",
        "period_set": "full record",
        "required": [
            p_diag("temporal_stats_OL_20000601_20240531.nc4"),
            p_diag("temporal_stats_DA_20000601_20240531.nc4"),
            p_diag("LS_OLv8_M36.ldas_tilecoord.bin"),
        ],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 5",
        "description": "Monthly normalized OmF evolution",
        "period_set": "fine_periods",
        "required": [
            p_diag("spatial_stats_OL_200006_202405.pkl"),
            p_diag("spatial_stats_DA_200006_202405.pkl"),
        ],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 6",
        "description": "OmF maps by period and sensor",
        "period_set": "fine_periods where available; P1/P2 need new temporal stats if plotted separately",
        "required": [p_diag("LS_OLv8_M36.ldas_tilecoord.bin")],
        "reference": [],
        "upstream": "projects/M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb",
    },
    {
        "figure": "Fig. 7",
        "description": "ISMN surface/root-zone soil moisture skill deltas",
        "period_set": "validation_periods",
        "required": [p_rel("projects/M21C_ls/output/ismn_network_skill/batch_figures/all_networks_hybrid_OL_DA_delta_surface_rz_R_anomR_ubRMSE_table.csv")],
        "reference": [p_rel("projects/M21C_ls/output/ismn_network_skill/batch_figures/all_networks_hybrid_OL_DA_delta_surface_rz_R_anomR_ubRMSE.png")],
        "upstream": "projects/M21C_ls/notebooks/insitu_skill_cached_batch_figures.ipynb",
    },
    {
        "figure": "Fig. 8",
        "description": "IMS snow-cover categorical skill deltas",
        "period_set": "full record",
        "required": [
            p_rel("projects/IMS/output/ims_ol_da_cell_counts_metrics_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.nc4"),
            p_rel("projects/IMS/output/ims_ol_da_comparison_table_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.csv"),
            p_rel("projects/IMS/output/ims_ol_da_scope_metadata_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10.csv"),
        ],
        "reference": [p_rel("projects/IMS/output/figures_ims_maps_and_tables/ims_all_period_delta_metrics_SMAP_EASEv2_M36_GLOBAL_2000_2024_thr0p50_imsSnowDaysGe10_nh_robinson_2x3.png")],
        "upstream": "projects/IMS/scripts/run_ims_ol_da_cell_metrics.py; projects/IMS/notebooks/ims_maps_and_tables_from_precomputed_outputs.ipynb",
    },
    {
        "figure": "Fig. 9",
        "description": "SNOTEL SWE validation",
        "period_set": "full record/seasons",
        "required": [
            p_rel("projects/SNOTEL/outputs_snotel_ol_da_validation/snotel_station_metrics_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.csv"),
            p_rel("projects/SNOTEL/outputs_snotel_ol_da_validation/tables/snotel_swe_toprow_bar_values_ci_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.csv"),
        ],
        "reference": [p_rel("projects/SNOTEL/outputs_snotel_ol_da_validation/figures/snotel_swe_2x3_bars_allsites_maps_da_minus_ol_elevfilt500_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.png")],
        "upstream": "projects/SNOTEL/notebooks/snow_daily_seasonal_ol_da_swe_snwd.ipynb",
    },
    {
        "figure": "Fig. 10",
        "description": "GHCN snow-depth validation",
        "period_set": "full record/seasons",
        "required": [p_rel("projects/GHCN_snwd/outputs_ghcn_snwd_ol_da_validation/ghcn_station_metrics_baseline_core_SMAP_EASEv2_M36_GLOBAL_20000101_20241231.csv")],
        "reference": [p_rel("projects/GHCN_snwd/outputs_ghcn_snwd_ol_da_validation/figures/ghcn_baseline_core_snodpland_2x3_bars_maps_nh_da_minus_ol_ALL_SMAP_EASEv2_M36_GLOBAL_20000101_20241231.png")],
        "upstream": "projects/GHCN_snwd/notebooks/ghcn_snwd_daily_seasonal_ol_da_snwd_baseline_basic.ipynb",
    },
    {
        "figure": "Fig. 11",
        "description": "ERA5-Land soil-moisture bars",
        "period_set": "validation_periods",
        "required": [
            p_rel("projects/era5_land/notebooks/ERA5L_vs_OLv8_M36_strict_summary.nc"),
            p_rel("projects/era5_land/notebooks/ERA5L_vs_DAv8_M36_strict_summary.nc"),
        ],
        "reference": [p_rel("projects/era5_land/notebooks/figures_era5l_bars/bars_surface_rz_sm_combined_3x3_era5l_only.png")],
        "upstream": "projects/era5_land/notebooks/plot_ERA5L_comparison_bars.ipynb",
    },
    {
        "figure": "Fig. 12",
        "description": "ERA5-Land soil-moisture maps",
        "period_set": "validation_periods",
        "required": [
            p_rel("projects/era5_land/notebooks/ERA5L_vs_OLv8_M36_strict_summary.nc"),
            p_rel("projects/era5_land/notebooks/ERA5L_vs_DAv8_M36_strict_summary.nc"),
            p_diag("LS_OLv8_M36.ldas_tilecoord.bin"),
        ],
        "reference": [p_rel("projects/era5_land/notebooks/figures_era5l_postage_stamps/postage_stamp_da_minus_ol_surface_rz_sm_6x3.png")],
        "upstream": "projects/era5_land/notebooks/plot_ERA5L_postage_stamp_maps.ipynb",
    },
    {
        "figure": "Fig. 13",
        "description": "ERA5-Land snow comparison",
        "period_set": "full record",
        "required": [
            p_rel("projects/era5_land/notebooks/ERA5L_vs_OLv8_M36_strict_summary.nc"),
            p_rel("projects/era5_land/notebooks/ERA5L_vs_DAv8_M36_strict_summary.nc"),
        ],
        "reference": [p_rel("projects/era5_land/notebooks/figures_era5l_bars/bars_snow_combined_3x3_era5l_only.png")],
        "upstream": "projects/era5_land/notebooks/plot_ERA5L_comparison_bars.ipynb",
    },
]

pd.DataFrame([{k: v for k, v in row.items() if k not in {"required", "reference"}} for row in figure_registry])

## Availability Checks

These checks intentionally run before any plotting. A missing file does not necessarily block every figure, but it should be resolved or explicitly accepted before rebuilding that figure.

In [ ]:
def file_status(path: Path) -> dict:
    p = Path(path)
    exists = p.exists()
    return {
        "path": str(p),
        "exists": exists,
        "size_mb": round(p.stat().st_size / 1024**2, 3) if exists and p.is_file() else np.nan,
        "mtime": pd.to_datetime(p.stat().st_mtime, unit="s") if exists else pd.NaT,
    }


availability_rows = []
for fig in figure_registry:
    for role in ("required", "reference"):
        paths = fig.get(role, [])
        if not paths:
            continue
        for path in paths:
            row = file_status(path)
            row.update({
                "figure": fig["figure"],
                "role": role,
                "description": fig["description"],
                "upstream": fig["upstream"],
            })
            availability_rows.append(row)

availability = pd.DataFrame(availability_rows)[[
    "figure", "role", "exists", "size_mb", "mtime", "description", "path", "upstream"
]]

missing_required = availability[(availability["role"] == "required") & (~availability["exists"])]

display(availability.sort_values(["figure", "role", "path"]))
if len(missing_required):
    print("Missing required products:")
    display(missing_required[["figure", "description", "path", "upstream"]])
else:
    print("All registered required products are present.")

## Fine-Period OFA Temporal-Stats Availability

Fig. 6 currently depends on precomputed period-split `temporal_stats_*` files. The Aqua transition can be shown from monthly count pickles, but separate P1/P2 OmF maps require matching period-split temporal stats if we want rows for those exact periods.

In [ ]:
def ymd(ts: pd.Timestamp) -> str:
    return ts.strftime("%Y%m%d")


period_stat_rows = []
for _, row in fine_periods.iterrows():
    for exp in ("OL", "DA"):
        path = p_diag(f"temporal_stats_{exp}_{ymd(row['start'])}_{ymd(row['end'])}.nc4")
        status = file_status(path)
        status.update({
            "period_id": row["period_id"],
            "period_label": row["label"],
            "exp": exp,
        })
        period_stat_rows.append(status)

period_stats_availability = pd.DataFrame(period_stat_rows)[[
    "period_id", "period_label", "exp", "exists", "size_mb", "mtime", "path"
]]
display(period_stats_availability)

missing_period_stats = period_stats_availability[~period_stats_availability["exists"]]
if len(missing_period_stats):
    print("Fine-period temporal_stats files missing. Fig. 6 can use existing coarse splits or these can be regenerated.")
    display(missing_period_stats[["period_id", "period_label", "exp", "path"]])
else:
    print("All fine-period temporal_stats files are present.")

## Next Plotting Cells

Add plotting cells only after deciding how to handle any missing period-split OFA products. The first plotting target should be Fig. 1, because it establishes period labels and shading used by later figures.